# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install `mlcroissant` if not already available
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and explore the high-level description with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("=== Dataset Overview ===")
print(f"Title         : {metadata.name}")
print(f"Description   : {metadata.description}\n")
print(f"Published     : {getattr(metadata, 'datePublished', None)}")
print(f"Version       : {getattr(metadata, 'version', None)}")
print(f"Identifier    : {getattr(metadata, 'identifier', None)}")

## 2. Data Overview
Review available record sets and the fields (columns) within each, along with their `@id` values for precise referencing.

In [ ]:
# List all available record sets with their @id
print("=== Record Sets ===")
record_sets = []
for rec in metadata.record_sets:
    print(f"- @id: {rec.id if hasattr(rec, 'id') else rec['@id']}, name: {rec.name if hasattr(rec, 'name') else rec.get('name')}")
    record_sets.append(rec.id if hasattr(rec, 'id') else rec['@id'])
    # List fields for each record set
    print("   Fields (by @id):")
    for field in rec.fields:
        fname = field.name if hasattr(field, 'name') else field.get('name')
        fid = field.id if hasattr(field, 'id') else field['@id']
        dtype = getattr(field, 'data_type', field.get('dataType', None))
        print(f"      - {fid}: {fname} [{dtype}]")
print("\nAll entity references in the analytics below use these @id values.")

## 3. Data Extraction
Load all records from each record set into pandas DataFrames by referencing their record set `@id`.

In [ ]:
# Extract data for each record set, referenced by @id
dataframes = {}
for rec_id in record_sets:
    print(f"Loading records for record set: {rec_id}")
    try:
        records = list(dataset.records(record_set=rec_id))
        dataframes[rec_id] = pd.DataFrame(records)
        print(f"  - Columns: {list(dataframes[rec_id].columns)}")
        print(f"  - Number of rows: {len(dataframes[rec_id])}\n")
    except Exception as e:
        print(f"  Warning: Unable to load data for {rec_id} ({e})")

# For demonstration, select the first record set for detailed EDA,
# but you can use any record_set @id discovered above.
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
    print(f"Proceeding with record set: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We now select numeric fields for further processing, such as outlier removal and normalization, referencing all columns by their `@id` as previously printed.

In [ ]:
# Inspect columns to pick a numeric field and a group/categorical field.
# Replace <numeric_field_id> and <group_field_id> with @id values shown before.
df = dataframes[main_record_set_id]
print("Available columns:", df.columns.tolist())

# For this dataset (protein analysis), likely numeric fields include 'MolecularWeight', 'NormalizedAbundance', etc.
# To demonstrate, suppose there is a field with @id 'molecularWeight' and group by 'sampleType' (change as needed per above output)

numeric_field = None
group_field = None
for col in df.columns:
    if 'weight' in col.lower() or 'abundance' in col.lower():
        numeric_field = col
    if 'sample' in col.lower():
        group_field = col
if not numeric_field:
    numeric_field = df.select_dtypes(include=['number']).columns[0] if len(df.select_dtypes(include=['number']).columns)>0 else df.columns[0]
if not group_field:
    group_field = df.columns[1] if len(df.columns)>1 else df.columns[0]

print(f"Chosen numeric field: {numeric_field}")
print(f"Chosen group field: {group_field}")

# Filtering: select values above an arbitrary threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} ({len(filtered_df)} rows):")
display(filtered_df.head())

# Normalization
filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized values for {numeric_field}:")
display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Grouping
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Mean of {numeric_field} grouped by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its normalized values, and compare group means. We'll use matplotlib for basic charts.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,3))
filtered_df[numeric_field].hist(bins=30)
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.title(f"Distribution of {numeric_field}")
plt.show()

plt.figure(figsize=(8,4))
filtered_df[numeric_field + '_normalized'].hist(bins=30)
plt.xlabel(numeric_field + " (Normalized)")
plt.ylabel('Frequency')
plt.title(f"Distribution of normalized {numeric_field}")
plt.show()

# Grouped means bar chart
if group_field in filtered_df.columns:
    plt.figure(figsize=(8,4))
    grouped_df.plot.bar(x=group_field, y=numeric_field, legend=False)
    plt.ylabel(f"Mean {numeric_field}")
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded the FAIR^2 Croissant dataset description and records using `mlcroissant`.
- Explored record sets and field structures using precise `@id` references.
- Loaded data into pandas DataFrames for analysis.
- Applied EDA steps: filtering, normalizing, aggregating, and grouped analysis.
- Visualized data distributions.

Refer to the dataset documentation and original Croissant schema for deeper field descriptions and advanced analytics.

All operations above use record set and field `@id`s, ensuring robust, explicit dataset referencing.